# KronLLM Q10 — 10M bilingual pretraining

One epoch over KronCorpus: the step count is derived from the
train bin, so the run always covers the whole corpus once.
(batch 24 × seq 512 × 2 T4).

Checkpoints every 500 steps to `/kaggle/working/q10`; the session stops at a
wall-clock budget so the GPU quota is not wasted, and the next session resumes
from `last.pt` attached as a dataset.

In [ ]:
import glob
import os
import subprocess
import sys
import zipfile

print(sorted(os.listdir('/kaggle/input')))
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

# The code dataset ships one zip: the kronllm package plus scripts/.
CODE = '/kaggle/working/code'
os.makedirs(CODE, exist_ok=True)
for z in glob.glob('/kaggle/input/kron-llm-code/kron-code.zip'):
    with zipfile.ZipFile(z) as fh:
        fh.extractall(CODE)
    print('unzipped', z)
sys.path.insert(0, CODE)

import kronllm
print('kronllm', kronllm.__file__)

DATA = '/kaggle/input/kron-corpus-2b'
print('bins', {p: round(os.path.getsize(os.path.join(DATA, p)) / 1e9, 2)
               for p in ('train.bpe.bin', 'val.bpe.bin')})

In [ ]:
import json
meta = json.load(open(DATA + '/tokenizer_meta.json'))
print('vocab', meta['vocab_size'], 'chat ids', meta.get('chat_tokens'))
print('train tokens', meta['train_tokens'], 'val tokens', meta['val_tokens'])
print('bytes/token', meta['bytes_per_token'])
print('token share', meta.get('token_share'), 'byte share', meta.get('byte_share'))

In [ ]:
cmd = ['torchrun', '--nproc_per_node=2', '--standalone',
       '-m', 'kronllm.cli', 'train', '--exp', 'Q10',
       '--data-dir', DATA, '--run-dir', '/kaggle/working/q10',
       '--override', 'training.max_minutes=500']

resume = sorted(glob.glob('/kaggle/input/*/last.pt'))
if resume:
    cmd += ['--resume', resume[0]]
    print('resuming from', resume[0])
print(' '.join(cmd))
env = dict(os.environ, PYTHONPATH=CODE)
subprocess.run(cmd, check=True, env=env)

In [ ]:
print(open('/kaggle/working/q10/results.json').read()[:1200])
for name in ('best.pt', 'last.pt', 'log.csv'):
    p = '/kaggle/working/q10/' + name
    if os.path.exists(p):
        print(name, round(os.path.getsize(p) / 1e6, 1), 'MB')
print('download last.pt and upload it as a dataset to resume next session')